# Tahoe 100M Dataset Processing

This notebook contains the script used to process the Tahoe Therapeutics Tahoe 100M Perturbation Atlas dataset. The parquet files from the dataset's [HuggingFace page](https://huggingface.co/datasets/tahoebio/Tahoe-100M) are aggregated in batches to create a pseudobulk count matrix. Drug treatement samples were compared to DMSO samples in matching cell lines using the characteristic direction method and gene sets were created from the resulting signatures, filtering for protein-coding genes.

The resulting dataset contains 3,766,626 asssociations between 18,900 drug perturbations and 8,288 genes.

In [ ]:
import datetime
from datasets import load_dataset
import fsspec
from itertools import islice
import json
import math
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import pubchempy as pcp
import re
import requests
import scanpy as sc
import scipy.sparse as sp
from scipy.sparse import csr_matrix
import scipy.spatial.distance as dist
import seaborn as sns
import sys
from tqdm import tqdm

# UMAP
from sklearn.feature_extraction.text import TfidfVectorizer
import anndata
from collections import OrderedDict

# Bokeh
from bokeh.io import output_notebook
from bokeh.plotting import figure, show, save, output_file
from bokeh.models import HoverTool, ColumnDataSource
from bokeh.palettes import Category20
output_notebook()

from IPython.display import display, HTML, Markdown
sys.setrecursionlimit(100000000)

In [ ]:
tahoe_100m_ds = load_dataset('vevotx/Tahoe-100M', streaming=True, split='train')

In [ ]:
gene_metadata = load_dataset("vevotx/Tahoe-100M", name="gene_metadata", split="train")
gene_metadata.to_csv('gene_metadata.tsv', sep='\t')
gene_vocab = {entry["token_id"]: entry["ensembl_id"] for entry in gene_metadata}

In [ ]:
sample_metadata = load_dataset("vevotx/Tahoe-100M","sample_metadata", split="train").to_pandas()
drug_metadata = load_dataset("vevotx/Tahoe-100M","drug_metadata", split="train").to_pandas()
cell_line_metadata = load_dataset("vevotx/Tahoe-100M","cell_line_metadata", split="train").to_pandas()

## Collect and Pseudo-bulk Counts

In [ ]:
def create_anndata_from_generator(generator, gene_vocab, sample_size=None):
    sorted_vocab_items = sorted(gene_vocab.items())
    token_ids, gene_names = zip(*sorted_vocab_items)
    token_id_to_col_idx = {token_id: idx for idx, token_id in enumerate(token_ids)}

    data, indices, indptr = [], [], [0]
    obs_data = []

    for i, cell in tqdm(enumerate(generator), total=sample_size):
        genes = cell['genes']
        expressions = cell['expressions']
        if expressions[0] < 0: 
            genes = genes[1:]
            expressions = expressions[1:]

        col_indices = [token_id_to_col_idx[gene] for gene in genes if gene in token_id_to_col_idx]
        valid_expressions = [expr for gene, expr in zip(genes, expressions) if gene in token_id_to_col_idx]

        data.extend(valid_expressions)
        indices.extend(col_indices)
        indptr.append(len(data))

        obs_entry = {k: v for k, v in cell.items() if k not in ['genes', 'expressions']}
        obs_data.append(obs_entry)

    expr_matrix = csr_matrix((data, indices, indptr), shape=(len(indptr) - 1, len(gene_names)))
    obs_df = pd.DataFrame(obs_data).astype(str)
    
    adata = anndata.AnnData(X=expr_matrix, obs=obs_df)
    adata.var.index = pd.Index(gene_names, name='ensembl_id').astype(str)

    return adata

### Combine Parquet Files
There are 3388 parquet files making up the dataset. In this step, aggregation is performed on 23 at a time to create 148 batches.

In [ ]:
def parquet_row_generator(parquet_urls):
    for url in parquet_urls:
        with fsspec.open(url, 'rb') as f:
            df = pd.read_parquet(f)
            for _, row in df.iterrows():
                yield row.to_dict()

num_batches = 150
batch_size = 3388 // num_batches + 1

for batch in range(0, 148):
    start, end = batch_size*batch, batch_size*(batch+1)
    print(f'Processing batch {batch}, files {start}-{end-1}')
    parquet_urls = [f'https://huggingface.co/datasets/tahoebio/Tahoe-100M/resolve/main/data/train-{num:05d}-of-03388.parquet' for num in range(start, end)]

    adata = create_anndata_from_generator(parquet_row_generator(parquet_urls), gene_vocab, sample_size=28225*len(parquet_urls))
    adata.obs = pd.merge(adata.obs, sample_metadata.drop(columns=["drug","plate"]), on="sample")

    group_keys = ['sample', 'cell_line_id']
    adata.obs['group'] = adata.obs[group_keys].astype(str).agg('_'.join, axis=1)
    grouped = adata.obs.groupby('group')

    bulk_expr_rows = []
    group_names = []

    for name, indices in grouped.indices.items():
        submatrix = adata.X[indices]
        group_sum = submatrix.sum(axis=0)

        if not sp.issparse(group_sum):
            group_sum = csr_matrix(group_sum)

        bulk_expr_rows.append(group_sum)
        group_names.append(name)

    bulk_matrix = sp.vstack(bulk_expr_rows)

    obs = adata.obs.groupby(['group'])[['sample', 'cell_line_id']].first()
    obs.index = obs.index.astype(str)
    var = pd.DataFrame(index = adata.var_names if hasattr(adata, 'var_names') else [f'gene_{i}' for i in range(bulk_matrix.shape[1])])

    bulk_adata = anndata.AnnData(X=bulk_matrix, obs=obs, var=var)
    bulk_adata.write(f"out/tahoe100m_parquet_bulked_batch{batch}.h5ad")

### Combine Batches
In this step, the 148 batches are combined into 5 aggregated files.

In [ ]:
# Replace bounds to group 30 smaller batches into 1 large batch 5 times
batches = range(90,120)
batch_files = [f"tahoe100m_parquet_bulked_batch{batch}.h5ad" for batch in batches ]

adatas = [anndata.read_h5ad(os.path.join("out", f)) for f in batch_files]

merged = anndata.concat(adatas, axis=0, join='outer', label='batch', keys=batch_files)
grouped = merged.obs.groupby('group')

bulk_expr_rows = []
group_names = []

for name, indices in tqdm(grouped.indices.items()):
    submatrix = merged.X[indices]
    group_sum = submatrix.sum(axis=0)

    if not sp.issparse(group_sum):
        group_sum = csr_matrix(group_sum)

    bulk_expr_rows.append(group_sum)
    group_names.append(name)

bulk_matrix = sp.vstack(bulk_expr_rows)

var_index = merged.var_names if hasattr(merged, 'var_names') else [f'gene_{i}' for i in range(bulk_matrix.shape[1])]

bulk_adata = anndata.AnnData(X=bulk_matrix, obs=obs, var=var)
bulk_adata.write_h5ad("out/tahoe100m_parquet_bulked_batch90-119_merged.h5ad")

### Form Final Count File
In this final step, the 5 aggregated files are collected into one matrix.

In [ ]:
batch_files = [
    'tahoe100m_parquet_bulked_batch0-29_merged.h5ad',
    'tahoe100m_parquet_bulked_batch30-59_merged.h5ad', 
    'tahoe100m_parquet_bulked_batch60-89_merged.h5ad',
    'tahoe100m_parquet_bulked_batch90-119_merged.h5ad', 
    'tahoe100m_parquet_bulked_batch120-147_merged.h5ad'
]

adatas = [anndata.read_h5ad(os.path.join("out", f)) for f in batch_files]

merged = anndata.concat(adatas, axis=0, join='outer', label='batch', keys=batch_files)
grouped = merged.obs.groupby('group')

bulk_expr_rows = []
group_names = []

for name, indices in tqdm(grouped.indices.items()):
    if isinstance(indices, int) or len(indices) == 1:
        idx = indices if isinstance(indices, int) else indices[0]
        group_sum = merged.X[idx]
        if not sp.issparse(group_sum):
            group_sum = csr_matrix(group_sum)
    else:
        submatrix = merged.X[indices]
        group_sum = submatrix.sum(axis=0)
        if not sp.issparse(group_sum):
            group_sum = csr_matrix(group_sum)

    bulk_expr_rows.append(group_sum)
    group_names.append(name)

var_index = merged.var.index

adatas = ''
merged = ''

bulk_matrix = sp.vstack(bulk_expr_rows)

In [ ]:
combined = pd.DataFrame(bulk_matrix.todense().T, index=var_index, columns=pd.Index(group_names, name="group")).astype(int)
#combined.to_csv('tahoe100M_pseudobulk.csv.gz', compression='gzip')
bulk_matrix = 0
combined

In [ ]:
dex_groups = pd.DataFrame(index=group_names)
dex_groups['Sample'] = dex_groups.index.map(lambda x: '_'.join(x.split('_')[:2]))
dex_groups['Cell Line'] = dex_groups.index.map(lambda x: '_'.join(x.split('_')[2:]))

In [ ]:
sample_metadata = load_dataset("vevotx/Tahoe-100M","sample_metadata", split="train").to_pandas()
dex_samples = sample_metadata[sample_metadata['drug']=='Dexamethasone']['sample']
dmso_samples = sample_metadata[sample_metadata['drug']=='DMSO_TF']['sample']

In [ ]:
dex_groups = dex_groups[(dex_groups['Sample'].isin(dex_samples)) | (dex_groups['Sample'].isin(dmso_samples))]
dex_groups.to_csv('dex_dmso_groups.tsv', sep='\t')
dex_groups

In [ ]:
dex_expression = combined[dex_groups.index]
dex_expression.to_csv('dex_dmso_counts.csv')
dex_expression

## Computing Signatures

In [ ]:
# Function for computing signatures with characteristic direction
from maayanlab_bioinformatics.normalization.filter import filter_by_expr
from maayanlab_bioinformatics.dge.characteristic_direction import characteristic_direction
def cd_signature(control, treatment, data, metadata, meta_class_column_name):
    
    ctrl_ids = metadata.loc[metadata[meta_class_column_name]==control, :].index.tolist() #control
    case_ids = metadata.loc[metadata[meta_class_column_name]==treatment,:].index.tolist() #case
    
    signature = characteristic_direction(
        data.loc[:, ctrl_ids], 
        data.loc[:, case_ids], 
        calculate_sig=True
    )
    signature = signature.sort_values("CD-coefficient", ascending=False)
    signature['Significance'] = signature['CD-coefficient'].apply(abs)
    
    return signature

In [ ]:
control_samples = sample_metadata[sample_metadata['drug']=='DMSO_TF']
drug_samples = sample_metadata[sample_metadata['drug']!='DMSO_TF']
drug_dict = sample_metadata.set_index('sample').groupby(level=0)['drug'].first().to_dict()
drug_sample_dict = drug_samples.groupby('drug')['sample'].agg(list).to_dict()
cell_line_dict = cell_line_metadata.set_index('Cell_ID_Cellosaur').groupby(level=0)['cell_name'].first().to_dict()

In [ ]:
gene_metadata = pd.read_csv('gene_metadata.tsv', sep='\t')
gene_symbols = gene_metadata.set_index('ensembl_id')['gene_symbol'].to_dict()
expr_df = combined#.copy()
expr_df.index = expr_df.index.map(gene_symbols)
expr_df

In [ ]:
meta_df = pd.DataFrame(index=combined.columns)
meta_df['Sample'] = meta_df.index.map(lambda x:'_'.join(x.split('_')[:2]))
meta_df['Drug'] = meta_df['Sample'].map(drug_dict)
meta_df['Cell Line'] = meta_df.index.map(lambda x:'_'.join(x.split('_')[2:]))
meta_df

In [ ]:
signature_dir = 'RNAseq_data/signatures'
meta_class_column_name = 'Class'
control_name = 'Control'
treatment_name = 'Treatment'

In [ ]:
from random import seed, sample
seed(42)
signatures = {}
for drug in tqdm(drug_sample_dict):
    drug = drug.replace('/','-')
    for cell_line in cell_line_dict:
        if not os.path.exists(f'RNAseq_data/signatures/{drug}_{cell_line}.tsv'): 
            sub_meta_df = meta_df[(meta_df['Drug']==drug) & (meta_df['Cell Line']==cell_line)].copy()
            shape = sub_meta_df.shape[0]
            if shape==0:
                continue
            sub_meta_df.loc[:,'Class'] = 'Treatment'
            sub_control_samples = sample(control_samples['sample'].tolist(), shape)
            sub_control_meta_df = meta_df[(meta_df['Sample'].isin(sub_control_samples)) & (meta_df['Cell Line']==cell_line)].copy()
            sub_control_meta_df.loc[:,'Class'] = 'Control'
            sub_meta_df = pd.concat([sub_meta_df, sub_control_meta_df])
            sub_expr_df = expr_df[sub_meta_df.index]
            sub_expr_df = filter_by_expr(sub_expr_df, group=sub_meta_df[meta_class_column_name].apply(lambda x: x==treatment_name))
            signatures[drug, cell_line] =  cd_signature(control_name, treatment_name, sub_expr_df, sub_meta_df, meta_class_column_name)
            signatures[drug, cell_line].to_csv(f'{signature_dir}/{drug}_{cell_line}.tsv', sep='\t')
        else:
            signatures[drug, cell_line] = pd.read_csv(f'RNAseq_data/signatures/{drug}_{cell_line}.tsv', sep='\t', index_col=0)['CD-coefficient'].rename(f'{drug}_{cell_line}')

In [ ]:
signatures = {}
for file in tqdm(os.listdir('RNAseq_data/signatures/')):
    if file.endswith('.tsv'):
        drug = '_'.join(file.split('_')[:2])
        cell_line = '_'.join(file.split('_')[2:]).replace('.tsv','')
        signatures[drug, cell_line] = pd.read_csv(f'RNAseq_data/signatures/{drug}_{cell_line}.tsv', sep='\t', index_col=0)['CD-coefficient'].rename(f'{drug}_{cell_line}')

In [ ]:
expr_df = ''
combined = ''

In [ ]:
signatures[('(R)-Verapamil (hydrochloride)','CVCL_0023')]

In [ ]:
signatures = pd.concat(signatures.values(), axis=1)
signatures

In [ ]:
signatures.to_csv('RNAseq_data/tahoe_cd_signatures.csv')

### Gene Set Creation

In [ ]:
gene_info = pd.read_csv('../../tables/Homo_sapiens.gene_info.gz', sep='\t', compression='gzip')
gene_info = gene_info[(gene_info['#tax_id']==9606) & (gene_info['type_of_gene']=='protein-coding')]
gene_info['Symbol'] = gene_info['Symbol'].str.upper()
symbols = set(gene_info['Symbol'])
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
gene_info

In [ ]:
synonyms = gene_info.copy()
synonyms['Synonyms'] = synonyms.apply(lambda x: x['Synonyms'].upper().split('|') + [x['Symbol']], axis=1)
synonyms = synonyms.explode('Synonyms')
synonyms['Synonyms'] = synonyms['Synonyms'].str.upper()
synonyms = synonyms[synonyms['Synonyms']!='-']
synonyms = synonyms.set_index('Synonyms')['Symbol'].to_dict()

In [ ]:
signatures.index = signatures.index.map(str.upper).map(synonyms)
signatures.columns = signatures.columns.map(lambda x: x.split('_')[0] + '_' + cell_line_dict[f'CVCL_{x.split("_")[-1]}'])
signatures = signatures.loc[signatures.index.dropna()]
signatures

In [ ]:
def threshold(n):
    if n > 0: 
        return 1
    elif n < 0: 
        return -1
    return n

In [ ]:
tahoe = pd.concat([
    signatures.groupby(level=0).mean().apply(lambda x: x[x>0].nlargest(100)).stack(),
    signatures.groupby(level=0).mean().apply(lambda x: x[x<0].nsmallest(100)).stack()
]).to_frame().reset_index()
tahoe.columns = ['Gene', 'Drug_CellLine', 'CD']
tahoe['Threshold'] = tahoe['CD'].apply(threshold)
tahoe

In [ ]:
tahoe[['Gene', 'Drug_CellLine']].nunique()

In [ ]:
sns.histplot(tahoe['CD'])

In [ ]:
tahoe.groupby(['Drug_CellLine', 'Threshold'])['Gene'].count().mean()

## Process Data for SQL Ingestion

### Resource

In [ ]:
(120, 'Tahoe Therapeutics', 'A resource providing large-scale perturbative single-cell expression maps', 'A resource providing large-scale perturbative single-cell expression maps', 'https://www.tahoebio.ai/', 18900, 1)

### Dataset

In [ ]:
#(id, name, name_without_resource, description, association, gene_set_description, gene_sets_description, attribute_set_description, positive_association, negative_association, is_signed, is_continuous_valued, last_updated, directory, num_page_views, resource_fk, measurement_fk, dataset_group_fk, attribute_type_fk, attribute_group_fk, evidence_type, evidence_group, measurement_bias, attribute_type_plural)
(185, 'Tahoe Therapeutics Tahoe 100M Perturbation Atlas', 'Single-cell expression data from drug perturbation across cell lines in over 95 million cells', 'gene-drug perturbation associations by differential expression of gene following drug perturbation', 'genes differentially expressed following the {0} drug perturbation from the Tahoe Therapeutics Tahoe 100M Perturbation Atlas dataset.', 'sets of genes differentially expressed following drug perturbation from the Tahoe Therapeutics Tahoe 100M Perturbation Atlas dataset.', 'drug perturbations changing expression of {0} gene from the Tahoe Therapeutics Tahoe 100M Perturbation Atlas dataset.', 'increased expression', 'decreased expression', 1, 1, '2025-07-07', 'tahoe100m', 0, 120, 16, 7, 8, 2, 'gene expression by RNA-seq', 'primary experimental data', 'high-throughput, data-driven', 'drug perturbations')

### Publication

In [ ]:
#(id, long_citation, short_citation, url, first_author_last_name, first_author_initials, journal_abbreviation, year, title, volume, pages)
(169, 'Zhang J, et al. (2025) Tahoe-100M: A Giga-Scale Single-Cell Perturbation Atlas for Context-Dependent Gene Function and Cellular Modeling. bioRxiv. 2025.02.20.639398v3', 'Zhang, bioRxiv, 2025', 'https://www.biorxiv.org/content/10.1101/2025.02.20.639398v3','Zhang', 'J', 'bioRxiv', 2025, 'Tahoe-100M: A Giga-Scale Single-Cell Perturbation Atlas for Context-Dependent Gene Function and Cellular Modeling', 2025, '02.20.639398v3')
(273, 185, 169)

### Naming Authority

In [ ]:
#(id, name, description, url, publication_fk)
(114, 'Tahoe Therapeutics', 'A resource providing large-scale perturbative single-cell expression maps', 'https://www.tahoebio.ai/', 169)

### Attribute

In [ ]:
#(id, name_from_naming_authority, id_from_naming_authority, url_from_naming_authority, naming_authority_fk)
index = 517445
attributefks = {}
drug_ids = drug_metadata.set_index('drug')['pubchem_cid'].dropna().astype(int).astype(str).to_dict()
cell_line_ids = cell_line_metadata.set_index('cell_name')['Cell_ID_Cellosaur'].to_dict()

for pert in tahoe['Drug_CellLine'].unique():
    drug = pert.split('_')[0]
    cl = pert.split('_')[1]
    drugid = f'CID_{drug_ids[drug]}' if drug in drug_ids else 'NA'
    clid = cell_line_ids[cl]
    print((index, pert, f'{drugid}_{clid}', f'{drug} perturbation in {cl} cell line', 114), end=',\n')
    attributefks[pert] = index
    index += 1

### Gene Set

In [ ]:
#(id, name_from_naming_authority, id_from_naming_authority, url_from_naming_authority, naming_authority_fk)
index = 138600000
genesetfks = {}

for pert in tahoe['Drug_CellLine'].unique():
    drug = pert.split('_')[0]
    cl = pert.split('_')[1]
    drugid = f'CID_{drug_ids[drug]}' if drug in drug_ids else 'NA'
    clid = cell_line_ids[cl]
    print((index, pert, f'{drugid}_{clid}', f'{drug} perturbation in {cl} cell line', 185, 2, attributefks[pert]), end=',\n')
    genesetfks[pert] = index
    index += 1

### Association

In [ ]:
gene_fks = pd.read_csv('../../tables/gene.csv', index_col=1)['id'].to_dict()
associations = tahoe.copy()
associations['Gene'] = associations['Gene'].map(gene_fks)
associations['Drug_CellLine'] = associations['Drug_CellLine'].map(genesetfks)
associations.columns = ['gene_fk', 'gene_set_fk', 'standardized_value', 'threshold_value']
associations = associations.reset_index(drop=True)
associations.index += 80000000
associations.to_csv('../../harmonizome-update/tahoe100m.csv')
associations

## Create Downloads

In [ ]:
output_path = 'downloads'

### Ternary Matrix

In [ ]:
ternarymatrix = pd.crosstab(tahoe['Gene'], tahoe['Drug_CellLine'], tahoe['Threshold'], aggfunc=max).replace(np.nan, 0).astype(int)
ternarymatrixT = ternarymatrix.T
ternarymatrix.to_csv(f'{output_path}/gene_attribute_matrix.txt.gz', sep='\t', compression='gzip')
ternarymatrix

### Gene-Attribute Edge List

In [ ]:
def construct_id(pert):
    drug = pert.split('_')[0]
    cl = pert.split('_')[1]
    drugid = f'CID_{drug_ids[drug]}' if drug in drug_ids else 'NA'
    clid = cell_line_ids[cl]
    return f'{drugid}_{clid}'

In [ ]:
geneids = gene_info.set_index('Symbol')['GeneID'].to_dict()
edgelist = tahoe.copy()
edgelist['Gene ID'] = edgelist['Gene'].apply(lambda x: geneids[x])
edgelist['Perturbation ID'] = edgelist['Drug_CellLine'].map(construct_id)
edgelist = edgelist[['Gene', 'Gene ID', 'Drug_CellLine', 'Perturbation ID', 'CD', 'Threshold']]
edgelist.columns = ['Gene', 'Gene ID', 'Drug Perturbation', 'Drug Perturbation ID', 'Standardized Value', 'Threshold Value']
edgelist.to_csv(f'{output_path}/gene_attribute_edges.txt.gz', sep='\t', compression='gzip')
edgelist

### Gene List

In [ ]:
geneslist = edgelist[['Gene', 'Gene ID']].drop_duplicates().reset_index(drop=True)
geneslist.to_csv(f'{output_path}/gene_list_terms.txt.gz', sep='\t', compression='gzip')
geneslist

### Attribute List

In [ ]:
attributeslist = edgelist[['Drug Perturbation', 'Drug Perturbation ID']].drop_duplicates().reset_index(drop=True)
attributeslist.to_csv(f'{output_path}/attribute_list_entries.txt.gz', sep='\t', compression='gzip')
attributeslist

### Up Gene Set Library

In [ ]:
with open(f'{output_path}/gene_set_library_up_crisp.gmt', 'w') as f:
    arr = ternarymatrix.reset_index(drop=True).to_numpy(dtype=np.int_)
    attributes = ternarymatrix.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*ternarymatrix.index[arr[:, i] == 1]])>= 5:
            print(attributes[i], '', *ternarymatrix.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Down Gene Set Library

In [ ]:
with open(f'{output_path}/gene_set_library_dn_crisp.gmt', 'w') as f:
    arr = ternarymatrix.reset_index(drop=True).to_numpy(dtype=np.int_)
    attributes = ternarymatrix.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*ternarymatrix.index[arr[:, i] == -1]])>= 5:
            print(attributes[i], '', *ternarymatrix.index[arr[:, i] == -1], sep='\t', end='\n', file=f)

### Up Attribute Set Library

In [ ]:
with open(f'{output_path}/attribute_set_library_up_crisp.gmt', 'w') as f:
    arr = ternarymatrixT.reset_index(drop=True).to_numpy(dtype=np.int_)
    genes = ternarymatrixT.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*ternarymatrixT.index[arr[:, i] == 1]])>= 5:
            print(genes[i], '', *ternarymatrixT.index[arr[:, i] == 1], sep='\t', end='\n', file=f)

### Down Attribute Set Library

In [ ]:
with open(f'{output_path}/attribute_set_library_dn_crisp.gmt', 'w') as f:
    arr = ternarymatrixT.reset_index(drop=True).to_numpy(dtype=np.int_)
    genes = ternarymatrixT.columns

    w, h = arr.shape
    for i in tqdm(range(h)):
        if len([*ternarymatrixT.index[arr[:, i] == -1]])>= 5:
            print(genes[i], '', *ternarymatrixT.index[arr[:, i] == -1], sep='\t', end='\n', file=f)

### Gene Similarity Matrix

In [ ]:
gene_similarity_matrix = dist.pdist(ternarymatrix.to_numpy(dtype=np.int_), 'cosine')
gene_similarity_matrix = dist.squareform(gene_similarity_matrix)
gene_similarity_matrix = 1 - gene_similarity_matrix

gene_similarity_matrix = pd.DataFrame(data=gene_similarity_matrix, index=ternarymatrix.index, columns=ternarymatrix.index)
gene_similarity_matrix.index.name = None
gene_similarity_matrix.columns.name = None
gene_similarity_matrix.to_csv(f'{output_path}/gene_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
gene_similarity_matrix

### Attribute Similarity Matrix

In [ ]:
attribute_similarity_matrix = dist.pdist(ternarymatrixT.to_numpy(dtype=np.int_), 'cosine')
attribute_similarity_matrix = dist.squareform(attribute_similarity_matrix)
attribute_similarity_matrix = 1 - attribute_similarity_matrix

attribute_similarity_matrix = pd.DataFrame(data=attribute_similarity_matrix, index=ternarymatrixT.index, columns=ternarymatrixT.index)
attribute_similarity_matrix.index.name = None
attribute_similarity_matrix.columns.name = None
attribute_similarity_matrix.to_csv(f'{output_path}/attribute_similarity_matrix_cosine.txt.gz', sep='\t', compression='gzip')
attribute_similarity_matrix

### Gene Attribute Standardized Matrix

In [ ]:
standardizedmatrix = signatures.loc[tahoe['Gene'].unique(), tahoe['Drug_CellLine'].unique()].groupby(level=0).mean()
standardizedmatrix.to_csv(f'{output_path}gene_attribute_matrix_standardized.txt.gz', sep='\t', compression='gzip')
standardizedmatrix

### Knowledge Graph Serialization

In [ ]:
nodes = {}
edges = []

for gene in geneslist.index:
    gene = geneslist.loc[gene]
    nodes[int(gene['Gene ID'])] = {
        "type":"gene",
        "properties": {
            "id":int(gene['Gene ID']),
            "label":gene['Gene']
        }}

for pert in attributeslist.index:
    pert = attributeslist.loc[pert]
    nodes[pert['Drug Perturbation ID']] = {
        "type":"drug perturbation",
        "properties": {
            "label":pert['Drug Perturbation'],
            "id":pert['Drug Perturbation ID']
        }}

for edge in edgelist.index:
    edge = edgelist.loc[edge]
    if edge['Threshold Value'] == 1:
        edges.append({
            "source": edge['Drug Perturbation ID'],
            "relation": "increases expression of",
            "target": int(edge['Gene ID']),
            "properties":{
                "id": edge['Drug Perturbation ID']+":"+str(edge['Gene ID']),
                "source_id": edge['Drug Perturbation ID'],
                "source_label":edge['Drug Perturbation'],
                "target_label":edge['Gene'],
                "target_id": int(edge['Gene ID']),
                "directed":True,
                "standardized": float(edge['Standardized Value']),
                "threshold":1
            }})
    else:
        edges.append({
            "source": edge['Drug Perturbation ID'],
            "relation": "decreases expression of",
            "target": int(edge['Gene ID']),
            "properties":{
                "id": edge['Drug Perturbation ID']+":"+str(edge['Gene ID']),
                "source_id": edge['Drug Perturbation ID'],
                "source_label":edge['Drug Perturbation'],
                "target_label":edge['Gene'],
                "target_id": int(edge['Gene ID']),
                "directed":True,
                "standardized": float(edge['Standardized Value']),
                "threshold":-1
            }})

#### RDF

In [ ]:
with open(output_path+'/kg_serializations/tahoe100m.rdf', 'w') as f:
    print('@prefix gene: <https://ncbi.nlm.nih.gov/gene/> .', file=f)
    print('@prefix RO: <http://purl.obolibrary.org/RO_> .', file=f)
    
    print('', file=f)
    for edge in edges:
        if edge['properties']['threshold'] == 1:
            print(f"<{edge['properties']['source_id']}>", 'RO:0003003', 'gene:'+str(edge['properties']['target_id']), end=' .\n', file=f)
        else:
            print(f"<{edge['properties']['source_id']}>", 'RO:0003002', 'gene:'+str(edge['properties']['target_id']), end=' .\n', file=f)

#### JSON

In [ ]:
with open(output_path+'/kg_serializations/tahoe100m.json', 'w') as f:
    serial = json.dump(
        {
            "Version":"1", 
            "nodes": nodes,
            "edges": edges
        }, indent=4, fp=f)

#### TSV

In [ ]:
nodeframe = pd.DataFrame(nodes).T
nodeframe['id'] = nodeframe['properties'].apply(lambda x: x['id'])
nodeframe['label'] = nodeframe['properties'].apply(lambda x: x['label'])
namespace = {'gene':'NCBI Entrez', 'drug perturbation':'Tahoe 100M'}
nodeframe['namespace'] = nodeframe['type'].apply(lambda x: namespace[x])
nodeframe = nodeframe.get(['namespace', 'id', 'label']).reset_index(drop=True)
nodeframe.to_csv(output_path+'/kg_serializations/tahoe100m_tsv/nodes.tsv', sep='\t')
nodeframe

In [ ]:
edgeframe = pd.DataFrame(edges)
edgeframe['standardized'] = edgeframe['properties'].apply(lambda x: x['standardized'])
edgeframe['threshold'] = edgeframe['properties'].apply(lambda x: x['threshold'])
edgeframe = edgeframe.get(['source', 'relation', 'target', 'standardized', 'threshold'])
edgeframe.to_csv(output_path+'/kg_serializations/tahoe100m_tsv/edges.tsv', sep='\t')
edgeframe

## Prepare Visualizations

### Gene-Attribute Clustermap

In [ ]:
sns.clustermap(ternarymatrix, cmap='seismic', center=0, figsize=(10,10), xticklabels=False, yticklabels=False)

### Gene-Gene Similarity Clustermap

In [ ]:
sns.clustermap(gene_similarity_matrix, cmap='seismic', center=0)

### Attribute-Attribute Similarity Clustermap

In [ ]:
sns.clustermap(attribute_similarity_matrix, cmap='seismic', center=0, xticklabels=False, yticklabels=False)

### UMAP

In [ ]:
def load_gmt(file):
    gmt = OrderedDict()
    for line in file:
        term, blank, *geneset = line.strip().split('\t')
        if 'consensus' not in term:
            term = f'{term}_up'
        gmt[term] = ' '.join(set(geneset))
    return gmt
libdict = load_gmt(open(f'{output_path}/gene_set_library_up_crisp.gmt', 'r'))
def load_gmt(file):
    gmt = OrderedDict()
    for line in file:
        term, blank, *geneset = line.strip().split('\t')
        gmt[f'{term}_down'] = ' '.join(set(geneset))
    return gmt
downlibdict = load_gmt(open(f'{output_path}/gene_set_library_dn_crisp.gmt', 'r'))
libdict.update(downlibdict)
scatterdir = 'images/'

In [ ]:
def process_scatterplot(libdict, nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1):
    print("\tTF-IDF vectorizing gene set data...")
    vec = TfidfVectorizer(max_df=maxdf, min_df=mindf)
    X = vec.fit_transform(libdict.values())
    print(X.shape)
    adata = anndata.AnnData(X)
    adata.obs.index = libdict.keys()

    print("\tPerforming Leiden clustering...")
    ### the n_neighbors and min_dist parameters can be altered
    sc.pp.neighbors(adata, n_neighbors=nneighbors, use_rep='X')
    sc.tl.leiden(adata, resolution=1.0)
    sc.tl.umap(adata, min_dist=mindist, spread=spread, random_state=42)

    new_order = adata.obs.sort_values(by='leiden').index.tolist()
    adata = adata[new_order, :]
    adata.obs['leiden'] = 'Cluster ' + adata.obs['leiden'].astype('object')

    df = pd.DataFrame(adata.obsm['X_umap'])
    df.columns = ['x', 'y']

    df['cluster'] = adata.obs['leiden'].values
    df['term'] = adata.obs.index
    df['genes'] = [libdict[l] for l in df['term']]

    return df

In [ ]:
def get_scatter_colors(df):
    clusters = pd.unique(df['cluster']).tolist()
    colors = list(Category20[20])[::2] + list(Category20[20])[1::2]
    color_mapper = {clusters[i]: colors[i % 20] for i in range(len(clusters))}
    return color_mapper

def get_scatterplot(scatterdf):
    df = scatterdf.copy()
    color_mapper = get_scatter_colors(df)
    df['color'] = df['cluster'].apply(lambda x: color_mapper[x])

    hover_emb = HoverTool(name="df", tooltips="""
        <div style="margin: 10">
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Gene Set:</span>
                <span style="font-size: 12px">@gene_set</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Coordinates:</span>
                <span style="font-size: 12px">(@x,@y)</span>
            <div style="margin: 0 auto; width:300px;">
                <span style="font-size: 12px; font-weight: bold;">Cluster:</span>
                <span style="font-size: 12px">@cluster</span>
            </div>
        </div>
    """)
    tools_emb = [hover_emb, 'pan', 'wheel_zoom', 'reset', 'save']

    plot_emb = figure(
        width=1000, 
        height=700, 
        tools=tools_emb
    )

    source = ColumnDataSource(
        data=dict(
            x = df['x'],
            y = df['y'],
            gene_set = df['term'],
            cluster = df['cluster'],
            colors = df['color'],
            label = df['cluster']
        )
    )

    # hide axis labels and grid lines
    plot_emb.xaxis.major_tick_line_color = None
    plot_emb.xaxis.minor_tick_line_color = None
    plot_emb.yaxis.major_tick_line_color = None
    plot_emb.yaxis.minor_tick_line_color = None
    plot_emb.xaxis.major_label_text_font_size = '0pt'
    plot_emb.yaxis.major_label_text_font_size = '0pt' 

    plot_emb.output_backend = "svg"    
    
    plot_emb.title = 'Gene Sets in the Tahoe Therapeutics Tahoe 100M Perturbation Atlas Library'
    plot_emb.xaxis.axis_label = "UMAP_1"
    plot_emb.yaxis.axis_label = "UMAP_2"
    plot_emb.xaxis.axis_label_text_font_style = 'normal'
    plot_emb.xaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_size = '18px'
    plot_emb.yaxis.axis_label_text_font_style = 'normal'
    plot_emb.title.align = 'center'
    plot_emb.title.text_font_size = '18px'
    
    s = plot_emb.scatter(
        'x', 
        'y', 
        size = 3, 
        source = source, 
        color = 'colors'
    )
    
    return plot_emb

In [ ]:
## defaults: nneighbors=30, mindist=0.1, spread=1.0, maxdf=1.0, mindf=1
scatter_df = process_scatterplot(libdict, nneighbors=150,mindist=0.1
     #,spread=1.5 
     ,maxdf=0.5
     ,mindf=2
)

# Display Scatter Plot
plot = get_scatterplot(scatter_df)
output_notebook()
show(plot)

In [ ]:
output_file(filename=f"{scatterdir}/umap.html", title = 'Gene Sets in Tahoe Therapeutics Tahoe 100M Perturbation Atlas Library')
save(plot)